# 01 — Introduction to Map Encryption

## Problem Statement

GPS coordinates are quasi-identifying even without names attached. A sequence of locations — home, workplace, hospital, place of worship — can uniquely fingerprint an individual. Standard field-level encryption (e.g., AES-ECB on the raw bytes) is not enough: spatial range queries against stored tile indices `(qxp, qyp)` can reveal areas of interest without ever decrypting, simply by observing which index values cluster within a bounding box.

This scheme addresses the problem with a four-step pipeline:

| Step | Name | What it does |
|------|------|--------------|
| 1 | **Project** | Convert (lat, lon) to Web Mercator (x, y) in metres |
| 2 | **Snap + Shuffle** | Round to a 250 m grid tile; permute tile indices with a Feistel PRP |
| 3 | **Lock** | AEAD-encrypt the sub-tile residual (rx, ry) |
| 4 | **Wobble** | Add per-record jitter for display; jitter_key only needed |

## Prior Art: Donut Geomasking and H3 Hex-Grid Binning

Before the cryptographic pipeline introduced in this notebook, two simpler approaches
represent the established state of practice in public health GIS:

- **Donut geomasking** (NB00a): replace each true location with a random point
  displaced 50–125 m in a random direction. Re-identification risk is ~1–3 % at a
  20 m threshold; spatial cluster structure remains visible. No key required.
- **H3 hex-grid binning** (NB00c): snap each location to the centroid of a
  hexagonal cell (~174 m edge at resolution 9). Provides k-anonymity within each
  cell at the cost of losing all within-cell precision. No key required.

Both approaches are demonstrated in NB00a and NB00c using the same cholera dataset.
This notebook builds on their limitations — no cryptographic guarantees, no tamper
detection, no key-based access control — to motivate the full four-step pipeline.

<div style="background:#f5faf9;border:1px solid #b8ddd8;border-radius:8px;padding:12px 14px 14px;margin:10px 0 22px;font-family:sans-serif;">
<div style="font-size:11px;color:#5a9e99;margin-bottom:10px;font-style:italic;">Overview — introduces all four pipeline steps</div>
<div style="display:flex;align-items:stretch;">
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:0;position:relative;z-index:4;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB02</div><div style="font-weight:700;font-size:13px;">① Project</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:3;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB03</div><div style="font-weight:700;font-size:13px;">② Snap+Shuffle</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:2;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB04</div><div style="font-weight:700;font-size:13px;">③ Lock</div></div>
    <div style="background:#2a9d8f;color:white;padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:1;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB05</div><div style="font-weight:700;font-size:13px;">④ Wobble</div></div>
</div>
</div>

## Learning Objectives

By the end of this notebook you will be able to:

1. **Identify** the four steps of the geographic coordinate encryption pipeline — Project, Snap + Shuffle, Lock, and Wobble — and the library function responsible for each .
2. **Explain** why raw GPS coordinates are quasi-identifying and why field-level encryption alone is insufficient to protect location privacy.
3. **Use** `MapEncryption.encode()`, `decode()`, and `render_coordinates()` to encrypt, display, and losslessly recover the 250 cholera death locations.
4. **Distinguish** the roles of `prp_key`, `aead_key`, and `jitter_key` in the three-tier key-privilege model.
5. **Assess** the privacy–utility tradeoff illustrated by comparing original and jitter-displaced positions on the linked dual map.

In [1]:
import secrets
import math
import numpy as np
import matplotlib.pyplot as plt

from map_encryption import (
    MapEncryption, SchemeParams, SCHEME_VERSION,
    _R_EARTH, _CHACHA_AVAILABLE,
)

print(f'Scheme version: {SCHEME_VERSION}')
print(f'AEAD backend: {"ChaCha20-Poly1305 (cryptography)" if _CHACHA_AVAILABLE else "XOR+HMAC-SHA256 (fallback)"}')

def metres_to_deg(spread_m, at_lat):
    """Approximate conversion from metres to degrees at a given latitude."""
    lat_deg = spread_m / 111_320
    lon_deg = spread_m / (111_320 * math.cos(math.radians(at_lat)))
    return lat_deg, lon_deg

# in production: load from secrets manager
MASTER_KEY = secrets.token_bytes(32)
enc = MapEncryption(MASTER_KEY, SchemeParams())
print(f'MapEncryption instance created with bin_size_m={enc.params.bin_size_m}')

Scheme version: 1
AEAD backend: ChaCha20-Poly1305 (cryptography)
MapEncryption instance created with bin_size_m=250


---
## Stage-by-Stage Pipeline Trace


The four pipeline stages are applied sequentially to a single coordinate — the Broadwick Street pump from the 1854 Soho cholera outbreak. Each stage prints its input and output so you can follow the transformation step by step. Subsequent notebooks devote an entire chapter to each stage; this trace gives you the complete picture first.

| Stage | Name | Internal function |
|-------|------|-------------------|
| 1 | **Project** — degrees → Web Mercator metres | `_project` |
| 2 | **Snap + Shuffle** — round to tile; permute tile indices | `_prp_encrypt` |
| 3 | **Lock** — AEAD-encrypt the sub-tile residual | `_AEAD.encrypt` |
| 4 | **Wobble** — add display jitter (jitter_key only) | `render_coordinates` |

In [1]:
# Broadwick Street pump — the index case location for the 1854 Soho cholera outbreak.
CENTER_LAT, CENTER_LON = 51.513341, -0.136668

from map_encryption import _project, _derive_keys, _prp_encrypt, _build_ad, _AEAD, SCHEME_VERSION
import struct

BIN  = enc.params.bin_size_m          # tile edge length in metres (default 250)
keys = _derive_keys(MASTER_KEY)        # three subkeys: prp_key, aead_key, jitter_key
tweak = MapEncryption.make_tweak(record_id=1, extra=b'nb01-demo')

# ── Stage 1: Project ─────────────────────────────────────────────────────────
# Convert geographic degrees to Web Mercator metres so every tile covers
# the same ground distance regardless of latitude.
x, y = _project(CENTER_LAT, CENTER_LON)
print('Stage 1 — Project')
print(f'  Input : lat={CENTER_LAT},  lon={CENTER_LON}')
print(f'  Output: x={x:,.2f} m,  y={y:,.2f} m  (Web Mercator)')

# ── Stage 2: Snap + Shuffle ──────────────────────────────────────────────────
# Round projected coordinates to the nearest tile centre (Snap).
# The sub-tile residual (rx, ry) holds the full GPS precision — it will be
# encrypted in Stage 3.
# Then permute the tile indices with the Feistel PRP (Shuffle) so the stored
# tile address reveals nothing about the true location without prp_key.
qx  = int(round(x / BIN))   # plaintext tile column
qy  = int(round(y / BIN))   # plaintext tile row
rx  = x - qx * BIN          # sub-tile east offset  (|rx| <= BIN/2 metres)
ry  = y - qy * BIN          # sub-tile north offset (|ry| <= BIN/2 metres)
qxp, qyp = _prp_encrypt(qx, qy, keys.prp_key, tweak, BIN, enc.params.prp_rounds)
print('\nStage 2 — Snap + Shuffle')
print(f'  Snap  : tile (qx={qx}, qy={qy}),  residual rx={rx:.2f} m, ry={ry:.2f} m')
print(f'  Shuffle: plaintext ({qx}, {qy}) -> shuffled (qxp={qxp}, qyp={qyp})  [stored in record]')

# ── Stage 3: Lock ────────────────────────────────────────────────────────────
# AEAD-encrypt the sub-tile residual (rx, ry) to produce ct_resid.
# The associated data (AD) binds the ciphertext to the plaintext tile address
# so that swapping ct_resid from one record into another fails authentication.
# The nonce must be unique per call — it is public and stored in the record.
nonce   = secrets.token_bytes(12)              # 12-byte random nonce, one per record
ad      = _build_ad(qx, qy, tweak)            # AD = plaintext tile (qx,qy) + tweak
payload = struct.pack('>dd', rx, ry)           # 16 bytes: two 64-bit floats
ct      = _AEAD(keys.aead_key).encrypt(nonce, payload, ad)
print('\nStage 3 — Lock')
print(f'  AD (hex)    : {ad.hex()}  ({len(ad)} bytes)')
print(f'  nonce (hex) : {nonce.hex()}  ({len(nonce)} bytes)')
print(f'  ct_resid    : {ct.hex()[:32]}...  ({len(ct)} bytes, stored in record)')

# ── Stage 4: Wobble ──────────────────────────────────────────────────────────
# Derive a display position by adding per-record uniform jitter to the
# shuffled tile centre.  Only jitter_key is required — the true location
# is not recoverable from display coordinates alone.
record_for_display = {
    'qxp': qxp, 'qyp': qyp, 'nonce': nonce,
    'tweak': tweak, 'version': SCHEME_VERSION,
}
display_lat, display_lon = enc.render_coordinates(record_for_display)
print('\nStage 4 — Wobble (display jitter)')
print(f'  Display: lat={display_lat:.6f},  lon={display_lon:.6f}')

# Verify the full round-trip using the public decode API
full_record = {**record_for_display, 'ct_resid': ct}
decoded = enc.decode(full_record)
err = max(abs(decoded[0] - CENTER_LAT), abs(decoded[1] - CENTER_LON))
print(f'\nRound-trip decode error: {err:.2e} degrees  (PASS)' if err < 1e-9
      else f'\nRound-trip decode error: {err:.2e} degrees  (FAIL)')


NameError: name 'enc' is not defined

## Record Field Reference

| Field | Type | Size | Is it secret? | Purpose |
|-------|------|------|---------------|---------|
| `qxp` | int | 4–8 bytes | No — but opaque | Shuffled tile x-index; reveals nothing without PRP key |
| `qyp` | int | 4–8 bytes | No — but opaque | Shuffled tile y-index; reveals nothing without PRP key |
| `nonce` | bytes | 12 bytes | No — public | Unique per encode call; seeds AEAD and display jitter |
| `ct_resid` | bytes | 32 bytes | **Yes** | AEAD ciphertext of (rx, ry); holds full GPS precision |
| `tweak` | bytes | variable | No — public | Record-ID context; binds ciphertext to this record |
| `version` | int | 1 byte | No | Scheme version for forward-compatible migration |

**Key point:** only `ct_resid` is truly secret. Everything else can be stored in plain sight — it reveals no location without the keys.

**Three subkeys, three access tiers.** The master key is never used directly. `_derive_keys` produces three independent subkeys via BLAKE2s, each labeled with a domain-separation string so a compromise of one reveals nothing about the others:

| Subkey | Pipeline stage | Access tier | What it can do |
|--------|---------------|-------------|----------------|
| `prp_key` | Stage 2 — Shuffle | Full-decode | Reverses the Feistel PRP to recover plaintext tile indices `(qx, qy)` |
| `aead_key` | Stage 3 — Lock | Full-decode | Decrypts `ct_resid` to recover the sub-tile residual `(rx, ry)` |
| `jitter_key` | Stage 4 — Wobble | Display only | Derives the display-jitter offset; cannot recover any location |

A display-tier system (e.g., a public map renderer) needs only `jitter_key`. Without `prp_key` and `aead_key` the original coordinate is unrecoverable. NB05 details how BLAKE2s derives all three from a single master secret.

In [ ]:
# Encode all 250 deaths; separately generate jitter-only displacement (no PRP shuffle)
# so both DualMap panels stay in Soho — PRP-shuffled display positions would scatter
# globally, making a side-by-side comparison impossible at a single zoom level.
# oef_orig / oef_jit: Folium JsCode callbacks that cross-link popup open/close events
# between the two panels by storing references in window._m1 / window._m2 keyed by FID.
import pandas as pd
import folium
import folium.plugins as fp
from folium.utilities import JsCode
import numpy as np
from map_encryption import _project, _unproject

deaths = pd.read_csv('data/cholera_deaths.csv')
pumps  = pd.read_csv('data/pumps.csv')

n = len(deaths)
lats = deaths['LAT'].values
lons = deaths['LON'].values
fids = deaths['FID'].values.tolist()
dcounts = deaths['DEATHS'].values.tolist()

records = [
    enc.encode(lats[i], lons[i],
               tweak=MapEncryption.make_tweak(record_id=int(fids[i]),
                                              extra=b'nb01-demo'))
    for i in range(n)
]

# Jitter-only displacement (no PRP shuffle) so both panels stay in Soho
J = enc._params.bin_size_m * enc._params.jitter_max_frac
rng = np.random.default_rng(seed=1854)
jit_xy = np.array([_project(lats[i], lons[i]) for i in range(n)])
jit_xy += rng.uniform(-J, J, jit_xy.shape)
jit_latlon = [_unproject(x, y) for x, y in jit_xy]

# Build GeoJSON feature collections
def death_feature(i, lat, lon, jlat, jlon):
    return {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": {
            "fid": int(fids[i]),
            "deaths": int(dcounts[i]),
            "lat": float(lat), "lon": float(lon),
            "jlat": float(jlat), "jlon": float(jlon),
        }
    }

orig_geojson = {"type": "FeatureCollection",
                "features": [death_feature(i, lats[i], lons[i],
                                           jit_latlon[i][0], jit_latlon[i][1])
                             for i in range(n)]}

jit_geojson  = {"type": "FeatureCollection",
                "features": [
                    {
                        "type": "Feature",
                        "geometry": {"type": "Point",
                                     "coordinates": [jit_latlon[i][1], jit_latlon[i][0]]},
                        "properties": {
                            "fid": int(fids[i]), "deaths": int(dcounts[i]),
                            "lat": float(lats[i]), "lon": float(lons[i]),
                            "jlat": float(jit_latlon[i][0]), "jlon": float(jit_latlon[i][1]),
                        }
                    }
                    for i in range(n)]}

# --- JavaScript: pointToLayer and onEachFeature for each panel ---
ptl_orig = JsCode("""
function(feature, latlng) {
    return L.circleMarker(latlng, {
        radius: 5, fillColor: '#2166ac',
        color: '#000', weight: 0.5, fillOpacity: 0.75
    });
}
""")

ptl_jit = JsCode("""
function(feature, latlng) {
    return L.circleMarker(latlng, {
        radius: 5, fillColor: '#d6604d',
        color: '#000', weight: 0.5, fillOpacity: 0.75
    });
}
""")

oef_orig = JsCode("""
function(feature, layer) {
    var fid = feature.properties.fid;
    window._m1 = window._m1 || {};
    window._m1[fid] = layer;
    var p = feature.properties;
    layer.bindPopup(
        '<b>FID ' + fid + '</b><br>' +
        'Deaths: ' + p.deaths + '<br>' +
        'Lat: ' + p.lat.toFixed(5) + ' Lon: ' + p.lon.toFixed(5) + '<br>' +
        '<em style="color:#2166ac">Original position</em>'
    , {maxWidth: 220});
    layer.on('mouseover', function() {
        layer.openPopup();
        if (window._m2 && window._m2[fid]) window._m2[fid].openPopup();
    });
    layer.on('mouseout', function() {
        layer.closePopup();
        if (window._m2 && window._m2[fid]) window._m2[fid].closePopup();
    });
}
""")

oef_jit = JsCode("""
function(feature, layer) {
    var fid = feature.properties.fid;
    window._m2 = window._m2 || {};
    window._m2[fid] = layer;
    var p = feature.properties;
    layer.bindPopup(
        '<b>FID ' + fid + '</b><br>' +
        'Deaths: ' + p.deaths + '<br>' +
        'Displaced Lat: ' + p.jlat.toFixed(5) + ' Lon: ' + p.jlon.toFixed(5) + '<br>' +
        'Original Lat: ' + p.lat.toFixed(5) + ' Lon: ' + p.lon.toFixed(5) + '<br>' +
        '<em style="color:#d6604d">Jitter-displaced (±62.5 m)</em>'
    , {maxWidth: 240});
    layer.on('mouseover', function() {
        layer.openPopup();
        if (window._m1 && window._m1[fid]) window._m1[fid].openPopup();
    });
    layer.on('mouseout', function() {
        layer.closePopup();
        if (window._m1 && window._m1[fid]) window._m1[fid].closePopup();
    });
}
""")

CENTER_LAT, CENTER_LON = 51.513341, -0.136668
dual = fp.DualMap(location=[CENTER_LAT, CENTER_LON], zoom_start=15,
                  tiles='cartodbpositron')

# Death markers — left (original) and right (jittered)
folium.GeoJson(
    orig_geojson,
    point_to_layer=ptl_orig,
    on_each_feature=oef_orig,
    name="Deaths (original)"
).add_to(dual.m1)

folium.GeoJson(
    jit_geojson,
    point_to_layer=ptl_jit,
    on_each_feature=oef_jit,
    name="Deaths (jitter-displaced)"
).add_to(dual.m2)

# Pump markers on both sides
pump_style = dict(radius=7, color='black', weight=1.5,
                  fill=True, fill_color='white', fill_opacity=1.0)
for _, p in pumps.iterrows():
    folium.CircleMarker([p.LAT, p.LON], tooltip=p.Street,
                        popup=f"<b>{p.Street} pump</b>", **pump_style).add_to(dual.m1)
    folium.CircleMarker([p.LAT, p.LON], tooltip=p.Street,
                        popup=f"<b>{p.Street} pump</b>", **pump_style).add_to(dual.m2)

# Panel labels
for side, label, colour in [(dual.m1, "Original positions", "#2166ac"),
                             (dual.m2, "Jitter-displaced (±62.5 m)", "#d6604d")]:
    folium.map.Marker(
        [51.5155, -0.148],
        icon=folium.DivIcon(html=(
            f'<div style="font-weight:bold;font-size:12px;color:{colour};'
            f'background:rgba(255,255,255,0.85);padding:2px 7px;'
            f'border-radius:4px;white-space:nowrap">{label}</div>'
        ))
    ).add_to(side)

print(f"DualMap ready — {n} deaths, {len(pumps)} pumps. "
      "Hover any blue point to see the linked red displacement on the right.")
dual

**Figure 1a.** Linked DualMap comparing 250 original cholera death locations (blue, left panel) with jitter-displaced display positions (red, right panel). Hovering over any marker opens matched popups on both sides showing the record FID and coordinates. White circles mark the eight John Snow pump locations. Pan and zoom are synchronised between panels.

## What Comes Next

- **NB02 — Coordinate Projection**: Deep-dive into the Web Mercator formula and why a projected metre-based grid is necessary for uniform tile sizing.
- **NB03 — Grid Snapping and the Feistel PRP**: How points are snapped to tiles and how the Feistel network shuffles tile indices without losing injectivity.
- **NB04 — Residual Encryption with AEAD**: How ChaCha20-Poly1305 protects the sub-tile offset and why tamper detection is essential.
- **NB05 — Key Derivation and Display Jitter**: How three independent subkeys are derived and how jitter prevents co-location fingerprinting.
- **NB06 — The Complete Pipeline**: End-to-end walkthrough using only the public API across 500 synthetic points.
- **NB07 — Security Properties and Limitations**: Honest inventory of what the scheme protects, what it does not, and directions for improvement.

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.

## Glossary

| Term | Definition |
|------|-----------|
| **GPS coordinate** | A (latitude, longitude) pair identifying a point on Earth's surface. |
| **Quasi-identifier** | A field that does not directly name a person but can be combined with other data to re-identify them; GPS coordinates are a strong quasi-identifier. |
| **Encode** | The full pipeline operation that converts a (lat, lon) pair into an encrypted record containing `qxp`, `qyp`, `nonce`, `ct_resid`, `tweak`, and `version`. |
| **Decode** | The inverse operation: recovers exact (lat, lon) from an encrypted record using the master key; returns `None` if any field has been tampered with. |
| **render_coordinates** | Returns an approximate display position for a record using only `jitter_key`; never exposes the exact GPS coordinate. |
| **Master key** | A 32-byte secret from which all three subkeys (`prp_key`, `aead_key`, `jitter_key`) are derived; should be stored in a KMS or HSM. |
| **Tweak** | A per-record context string (record ID + optional extra bytes) that binds a ciphertext to its intended use; changing the tweak causes decryption to fail. |
| **Record** | The encrypted data structure produced by `encode()`; contains no plaintext GPS coordinates. |
| **Pipeline** | The four-step sequence — Project → Snap+Shuffle → Lock → Wobble — that transforms a GPS coordinate into an encrypted, display-ready record. |